# Hito 1 — Baseline de recuperación dispersa (TF-IDF y BM25)

**Proyecto Final · Procesamiento del Lenguaje Natural · Máster en IA Aplicada (UC3M)**

Este notebook implementa el baseline del Hito 1:

1. Carga del **corpus** y preprocesado con **spaCy** (lematización, filtrado POS, stopwords, frecuencia documental).
2. Conjunto de **consultas de evaluación** *known-item* (`queries.jsonl`).
3. Baselines de recuperación **dispersa**: **TF-IDF + coseno** y **BM25**.
4. Evaluación con **Recall@K (K=5, 10)** y **MRR@10**, y análisis de los fallos.

> **Corpus**: este notebook se ejecuta sobre el corpus **Food.com** (`data/corpus_foodcom.jsonl`,
> 10.000 recetas, muestreadas de forma estratificada por categoría para asegurar variedad).
> Las consultas (`data/queries_foodcom.jsonl`) se generaron con `llama3.1:8b` y se revisaron
> a mano (ver `REVISION_consultas.md`). Para usar otro corpus, cambiad la celda de configuración.


## 0. Configuración

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

# === CONFIGURACIÓN — cambia esto para usar otro corpus ===
CORPUS_PATH   = "data/corpus_foodcom.jsonl"  # JSONL con {"doc_id","text"} por línea
QUERIES_PATH  = "data/queries_foodcom.jsonl" # JSONL con {"qid","query","relevant_ids"}
SPACY_MODEL   = "en_core_web_sm"             # es_core_news_sm para corpus en español
KS            = (5, 10)                        # valores de K para Recall@K
BM25_K1, BM25_B = 1.5, 0.75                   # parámetros BM25 (repórtalos en el MILESTONE)
MIN_DF        = 2                             # frecuencia documental mínima de un término
MAX_DF_RATIO  = 0.5                           # ratio máximo de documentos (evita términos ubicuos)
# =======================================================
print("Config OK")

Config OK


## 1. Carga del corpus

In [2]:
from src.corpus import load_jsonl
docs = load_jsonl(CORPUS_PATH)
doc_ids = [d["doc_id"] for d in docs]
print(f"{len(docs)} documentos cargados")
print("Ejemplo:", docs[0]["doc_id"], "->", docs[0]["text"][:120], "...")

10000 documentos cargados
Ejemplo: recipe_60627 -> Rabbit Paprika
Ingredients: butter, red onion, bay leaf, sweet paprika, caraway seed, white pepper, black pepper, cayenn ...


## 2. Preprocesado con spaCy

La **misma** función se aplica a documentos y a consultas (requisito del enunciado:
*"las consultas deben preprocesarse exactamente igual que los documentos"*).
Pasos: tokenización, **lematización**, filtrado por **categoría gramatical** (nos
quedamos con sustantivos, nombres propios, verbos, adjetivos y adverbios),
eliminación de **stopwords** y de tokens no alfabéticos.

In [3]:
from src.preprocess import load_nlp, preprocess, preprocess_many
nlp = load_nlp(SPACY_MODEL)
corpus_tokens = preprocess_many([d["text"] for d in docs], nlp)

ejemplo = docs[0]
print("ORIGINAL:", ejemplo["text"])
print("LEMAS   :", preprocess(ejemplo["text"], nlp))

ORIGINAL: Rabbit Paprika
Ingredients: butter, red onion, bay leaf, sweet paprika, caraway seed, white pepper, black pepper, cayenne pepper, rabbit, salt, sour cream
Instructions: Melt 4 tblsps of the butter in a Dutch oven over high heat. When foam subsides, add onion slices, Saute, stirring frequently, until onion begins to wilt, about 2 minutes. Add 1/3 cup of stock and bay leaf; heat to boiling. Reduce heat to very low; cook, uncovered, stirring occasionally, for 30 minutes. Combine 1 tsp of paprika, caraway seeds and 3 peppers in a small bowl. Melt remaining 3 tblsps butter in large skillet over medium-high heat. When foam subsides, saute rabbit, in 2 batches if necessary, 3 minutes on each side. Sprinkle pieces evenly with spice mixture. Reduce heat to medium; continue to saute until browned, about 5 minutes. Transfer rabbit to casserole with onions. Pour remaining stock in skillet; heat to boiling, scraping bottom and sides of skillet to loosen browned bits. Add contents of skill

## 3. Vocabulario por frecuencia documental

Descartamos términos demasiado **raros** (df < `MIN_DF`) y demasiado **comunes**
(df > `MAX_DF_RATIO`·N). Documentos y consultas se restringen a este vocabulario,
de modo que las palabras fuera de vocabulario (OOV) se ignoran en ambos baselines.

In [4]:
from src.baselines import build_vocabulary, apply_vocabulary, document_frequencies
vocab = build_vocabulary(corpus_tokens, min_df=MIN_DF, max_df_ratio=MAX_DF_RATIO)
corpus_tokens_f = [apply_vocabulary(t, vocab) for t in corpus_tokens]
print(f"Vocabulario: {len(vocab)} términos")

# La misma tubería para una consulta:
def pp(text):
    return apply_vocabulary(preprocess(text, nlp), vocab)

import collections
df = document_frequencies(corpus_tokens)
print("Términos más frecuentes:", df.most_common(8))

Vocabulario: 5842 términos


Términos más frecuentes: [('instruction', 9988), ('minute', 7136), ('add', 6714), ('ingredient', 5945), ('salt', 5847), ('ingredients', 5654), ('pepper', 5167), ('heat', 5075)]


## 4. Baselines de recuperación dispersa

In [5]:
from src.baselines import TfidfRetriever, BM25Retriever
tfidf = TfidfRetriever(doc_ids).fit(corpus_tokens_f)
bm25  = BM25Retriever(doc_ids, k1=BM25_K1, b=BM25_B).fit(corpus_tokens_f)
print("TF-IDF: vocabulario del vectorizador =", len(tfidf.vectorizer.vocabulary_))
print(f"BM25  : k1={BM25_K1}, b={BM25_B} (implementación: rank_bm25.BM25Okapi)")

# Ejemplo de búsqueda:
q = "chicken curry with coconut milk and rice"
print("\nConsulta:", q)
print("TF-IDF  ->", tfidf.search(pp(q), 3))
print("BM25    ->", bm25.search(pp(q), 3))

TF-IDF: vocabulario del vectorizador = 5842
BM25  : k1=1.5, b=0.75 (implementación: rank_bm25.BM25Okapi)

Consulta: chicken curry with coconut milk and rice
TF-IDF  -> [('recipe_458700', 0.6431417576608345), ('recipe_74912', 0.6061453649891193), ('recipe_407690', 0.5114079965741846)]
BM25    -> [('recipe_541331', 20.620360014779898), ('recipe_37041', 19.123431367928912), ('recipe_74912', 18.801736245947637)]


## 5. Conjunto de consultas de evaluación (`queries.jsonl`)

Consultas *known-item*: para documentos elegidos al azar, un **LLM** redacta la
consulta que haría un usuario para encontrarlos, **sin copiar frases**. El
documento de origen es el relevante. Formato: `{"qid","query","relevant_ids"}`.

> Aquí cargamos el `queries.jsonl` ya generado y revisado. Se generó con
> `src/generate_queries.py` (celda siguiente, desactivada por defecto) y la **revisión
> manual** está documentada en `REVISION_consultas.md`; el cribado de apoyo (solape con
> la receta y con su título) se reproduce con `python -m src.review_queries`.

In [6]:
from src.generate_queries import load_queries
queries = load_queries(QUERIES_PATH)
print(f"{len(queries)} consultas cargadas")
for q in queries[:3]:
    print(" ", q["qid"], "|", q["query"], "->", q["relevant_ids"])

34 consultas cargadas
  q01 | Coconut based no bake energy bars with flax seed and dark chocolate topping -> ['recipe_479373']
  q02 | Fried Jewish fish patties made with ground fish and matzo meal, served on a bed of greens. -> ['recipe_60179']
  q03 | Grilled Middle Eastern style chicken kebabs with vegetables marinating in a spicy citrus sauce -> ['recipe_170046']


### 5.b (Opcional) Generar las consultas con un LLM

Pon `RUN_LLM = True` y configura el endpoint. Las consultas de este proyecto se generaron
con `model="llama3.1:8b"` en Ollama local (`http://localhost:11434/api/generate`); el
mismo modelo está en el servidor de la UC3M (`url="https://yiyuan.tsc.uc3m.es/api/generate"`
y tu `api_key`). Se generan 35 para poder descartar en la revisión manual y quedarse con ≥30.
La temperatura es 0,7, así que una nueva ejecución no produce exactamente las mismas consultas.

In [7]:
RUN_LLM = False  # ponlo a True para generar consultas con el LLM

if RUN_LLM:
    from src.generate_queries import generate_queries, print_for_review, save_queries
    gen = generate_queries(
        docs, n_queries=35, seed=11235,
        model="llama3.1:8b",
        url="http://localhost:11434/api/generate",
        # api_key="sk-alumnos-...",  # para el endpoint de la UC3M
    )
    print_for_review(gen, n=10)          # REVISIÓN MANUAL (≥10): documenta los descartes
    save_queries(gen, "data/queries_generadas.jsonl")
else:
    print("Generación por LLM desactivada (RUN_LLM=False). Usando queries.jsonl precargado.")

Generación por LLM desactivada (RUN_LLM=False). Usando queries.jsonl precargado.


## 6. Evaluación: Recall@K y MRR@10

Con una sola receta relevante por consulta, Recall@K es la fracción de consultas cuya receta
aparece en el top-K. El MRR se calcula sobre los 10 resultados recuperados (una receta en la
posición 11 o peor cuenta 0), por eso se reporta como **MRR@10**.

In [8]:
import pandas as pd
from src.evaluation import evaluate

s_tfidf, r_tfidf = evaluate(tfidf, queries, pp, ks=KS)
s_bm25,  r_bm25  = evaluate(bm25,  queries, pp, ks=KS)

tabla = pd.DataFrame([s_tfidf, s_bm25], index=["TF-IDF + coseno", "BM25"])
tabla

,n_queries,MRR@10,Recall@5,Recall@10
TF-IDF + coseno,34,0.3958,0.5,0.5294
BM25,34,0.3680,0.5,0.5882


## 7. Inspección por consulta

In [9]:
filas = []
for rt, rb in zip(r_tfidf, r_bm25):
    filas.append({
        "qid": rt.qid,
        "query": rt.query[:48],
        "rel": ",".join(rt.relevant_ids),
        "rank_TFIDF": rt.first_relevant_rank,
        "rank_BM25": rb.first_relevant_rank,
    })
pd.DataFrame(filas)

,qid,query,rel,rank_TFIDF,rank_BM25
0,q01,Coconut based no bake energy bars with flax seed,recipe_479373,2.0,2.0
1,q02,Fried Jewish fish patties made with ground fish,recipe_60179,1.0,1.0
2,q03,Grilled Middle Eastern style chicken kebabs with,recipe_170046,NaN,NaN
3,q04,Creamy sauce with pasta and sautéed vegetables b,recipe_478609,NaN,NaN
4,q05,Greek-inspired pasta salad with a tangy cheese a,recipe_503003,NaN,NaN
5,q06,Roasted garlic potato recipe with rosemary that',recipe_367050,1.0,1.0
6,q07,Beef and mushroom dish with creamy sauce served,recipe_30159,8.0,7.0
7,q08,Boiled egg salad sandwich recipe with creamy dre,recipe_235097,1.0,2.0
8,q09,Blackened fish recipe with Creole seasoning and,recipe_385182,1.0,1.0
9,q10,baked puff pastry cups filled with creamy aspara,recipe_455645,NaN,NaN


## 8. Análisis de fallos

Consultas en las que **ambos** métodos fallan (la receta relevante no entra en el top-10).
Para cada una se muestra en qué posición queda realmente la receta en el ranking completo,
qué términos de la consulta (ya preprocesados) están o no en la receta, y qué recetas
devuelve TF-IDF en su lugar.

In [10]:
from src.evaluation import failure_analysis, diagnose_failures
pd.set_option("display.max_colwidth", None)

fails = failure_analysis({"TF-IDF": r_tfidf, "BM25": r_bm25}, k=max(KS))
print(f"Consultas que fallan en ambos (top-{max(KS)}): {len(fails)} de {len(queries)}")

doc_tokens = dict(zip(doc_ids, corpus_tokens_f))
doc_titles = {d["doc_id"]: d["text"].splitlines()[0].strip() for d in docs}
diag = diagnose_failures(fails, {"TF-IDF": tfidf, "BM25": bm25}, pp, doc_tokens, doc_titles)
pd.DataFrame(diag).set_index("qid")

Consultas que fallan en ambos (top-10): 14 de 34


,query,receta,rank_TF-IDF,rank_BM25,en_receta,no_en_receta,devuelve_en_su_lugar
qid,,,,,,,
q03,Grilled Middle Eastern style chicken kebabs with vegetables marinating in a spicy citrus sauce,Shish Taouk - Chicken Kebabs,59,60,"[chicken, vegetable, marinate]","[grilled, middle, eastern, style, kebab, spicy, citrus, sauce]","[Egyptian Fish Kebabs, Chicken & Kebab Grilled Veg Mix!, Middle Eastern Spice Mix]"
q04,Creamy sauce with pasta and sautéed vegetables baked with extra cheese topping,Cream Cheese Sauce,92,190,"[sauce, pasta, bake, cheese]","[creamy, sautéed, vegetable, extra, topping]","[Mac N Cheese, North Croatian Creamy Vegetable Soup, Italian Style Pasta Pan]"
q05,Greek-inspired pasta salad with a tangy cheese and olive twist,"Spaghetti With Tomatoes, Black Olives, Garlic, and Feta Cheese",314,300,"[pasta, cheese, olive]","[greek, inspire, salad, tangy, twist]","[Greek Pasta Salad, Turkey Twist Pasta Salad, Greek Pasta Salad]"
q10,baked puff pastry cups filled with creamy asparagus and chicken mixture,Chicken and Asparagus Vol-Au-Vents,70,49,"[bake, asparagus, chicken]","[puff, pastry, cup, fill, creamy, mixture]","[Creamy Lemon Asparagus Sauce, Om Ali, Asparagus Risotto]"
q13,Smooth blended tropical drink with banana and fruit juices for summer.,Paradise Freeze,143,90,"[smooth, banana, juice]","[blend, tropical, drink, fruit, summer]","[Raspberry Banana Mango Smoothie, Tropical Breakfast, Luscious Green Brekkie-On-The-Go!]"
q14,Greek pasta salad with tangy dressing and crumbled white cheese.,Greek Orzo Salad,26,20,"[greek, pasta, salad, cheese]","[tangy, dressing, crumble, white]","[Greek Pasta Salad, Greek Pasta Salad, Pasta Salad]"
q15,"Island-style stuffed egg halves with a creamy, peppery yolk mix",Caribbean Deviled Eggs,20,36,"[egg, half, yolk]","[island, style, stuff, creamy, peppery, mix]","[Peppery Roasted Squash Pate, Rowand Style Deviled Eggs, Dees Plain Old Deviled Eggs]"
q18,Homemade spice blend recipe for traditional East African red pepper sauce mix.,Eritrean Berbere (Red Pepper Spice Mixture),121,163,"[spice, red, mix]","[homemade, blend, recipe, traditional, east, african, sauce]","[North African Spice Blend, Homemade Pumpkin Pie Spice, North African Ras El Hanout Spice Mix]"
q22,Low-calorie Italian pasta dish with creamy sauce and fresh herbs.,Skinny Fettuccine Alfredo,1120,1269,"[low, sauce, fresh]","[calorie, italian, pasta, dish, creamy, herb]","[Amazing Tomato and Basil Penne With Italian Sausage, Rigatoni With Meat and Cheese Sauce, John's Bowtie Pasta Salad]"


### Por qué fallan

Las 14 consultas fallidas se agrupan en cuatro causas (una consulta puede tener varias):

**1. Sinónimos e hiperónimos (brecha semántica): la causa principal.** La consulta nombra el
plato o los ingredientes con otras palabras y el solape léxico desaparece:
- **q22** *"Low-calorie Italian pasta dish with creamy sauce"* busca *Skinny Fettuccine Alfredo*:
  la receta dice *fettuccine*, nunca *pasta*, *creamy* ni *Italian*. Queda en el puesto ~1.100.
- **q05** *"Greek-inspired pasta salad… tangy cheese and olive"* busca unos espaguetis con
  *feta* y aceitunas: solo comparte *pasta*, *cheese* y *olive*, y las dos recetas llamadas
  *Greek Pasta Salad* encajan mucho mejor. Puesto ~300.
- **q10** *puff pastry cups* = *vol-au-vents*; **q15** *stuffed egg halves* = *deviled eggs*;
  **q24** *organ meat… Louisiana* = *liver… Creole*; **q34** *seafood* = *shrimp, clams* y
  *pasta* = *noodles*.

**2. Consultas correctas pero genéricas, con competidores más "típicos".** La consulta
describe bien la receta, pero con términos que comparten muchas otras:
- **q14** *Greek pasta salad*: la receta (*Greek Orzo Salad*) queda en el puesto 20–26,
  detrás de varias recetas literalmente llamadas *Greek Pasta Salad*.
- **q23** *baked ham glazed…*: ganan *Apricot Glazed Baked Ham* y *Baked Ham With Bourbon
  Glaze*; la receta buscada (un jamón asado en la chimenea) no usa la palabra *glaze*.
- **q13**, **q18**, **q29**, **q30** y **q04** siguen el mismo patrón (batidos tropicales,
  mezclas de especias africanas, guisos en olla lenta, cazuelas de tacos). En **q04** además la
  receta es una salsa genérica y la consulta describe uno de sus usos posibles.

**3. Lematización inconsistente.** El modelo pequeño de spaCy no lematiza igual un participio
según su contexto: *"Grilled chicken"* da `grill`, pero *"Baked ham"* da `baked`. En **q03** la
consulta queda con `grilled` y la receta con `grill`; en **q23**, `baked` frente a `bake`.
Es un fallo del preprocesado, no del método, y sí se puede corregir en el baseline.

**4. Filtro de frecuencia documental.** `pepper` aparece en más del 50 % de las recetas y se
elimina del vocabulario. En **q18** (*red pepper* → *berbere*) y **q24** era un término útil.

**Conclusión.** Las causas 1 y 2 necesitan comparar **significados**, no palabras: es lo que
debe aportar la recuperación densa del Hito 2. Las causas 3 y 4 son mejorables dentro del
propio baseline (normalizar participios, ajustar `max_df`).

## 9. Conclusiones y siguiente paso

- Los baselines **dispersos** (TF-IDF y BM25) recuperan por **solape léxico**: funcionan
  bien cuando la consulta comparte términos con la receta y fallan ante **sinónimos o
  vocabulario distinto** (ver §8).
- Con 34 consultas, cada acierto vale ~0,03 puntos: las diferencias entre TF-IDF y BM25
  son de una o dos consultas y no permiten declarar un ganador.
- Estas métricas (**Recall@K**, **MRR@10**) son la **referencia** con la que se compara el
  sistema **RAG** del Hito 2.
- **Siguiente paso (Hito 2)**: recuperación **densa** con *sentence embeddings* y una base
  vectorial, manteniendo la **receta completa** como unidad de indexación y evaluada con
  **el mismo `queries.jsonl`** para una comparación justa; y primera evaluación de la
  **generación** con un LLM de pesos abiertos.

### Cómo reproducir
1. Instala el entorno (ver `README_Hito1.md`).
2. `Kernel > Restart & Run All` (o `jupyter nbconvert --to notebook --execute --inplace hito1_baseline.ipynb`).
3. Para regenerar las consultas: celda §5.b con `RUN_LLM=True`, cribado con
   `python -m src.review_queries` y revisión manual en `REVISION_consultas.md`.